In [14]:
import torch
import transformers
import pandas as pd
from tqdm import tqdm
transformers.__version__, torch.__version__

('4.46.3', '2.1.1')

In [15]:
# Device and paths:
device = 'cuda'
database_dir = '/scratch3/nic261/database/cxrmate_ed'  # The Hugging Face dataset will be saved here.

# Download model checkpoint:
model = transformers.AutoModelForCausalLM.from_pretrained('aehrc/cxrmate-ed', trust_remote_code=True).to(device=device)
tokenizer = transformers.PreTrainedTokenizerFast.from_pretrained('aehrc/cxrmate-ed')

A new version of the following files was downloaded from https://huggingface.co/aehrc/cxrmate-ed:
- configuration_cxrmate_ed.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modelling_cxrmate_ed.py:   0%|          | 0.00/62.9k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/aehrc/cxrmate-ed:
- utils.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/aehrc/cxrmate-ed:
- section_parser.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/aehrc/cxrmate-ed:
- create_section_files.py
- section_parser.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/aehrc/cxrmate-ed:
- prepare_dataset.py
- create_section_files.py
. Make sure to double-check they d

In [16]:
test_set = model.get_dataset_all_test_set_studies(database_dir=database_dir)

Loading dataset from disk:   0%|          | 0/1199 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/17 [00:00<?, ?it/s]

In [17]:
generated = []
for i in tqdm(range(len(test_set))):
    example = test_set[i]
    example = {k: v.to(device).unsqueeze(0) if isinstance(v, torch.Tensor) else [v] for k, v in example.items()}  # Add mini-batch dimension and move to device.

    # Convert the patient data in the batch into embeddings:
    inputs_embeds, attention_mask, token_type_ids, position_ids, bos_token_ids = model.prepare_inputs(tokenizer=tokenizer, **example)
        
    # Generate reports:
    output_ids = model.generate(
        input_ids=bos_token_ids,
        decoder_inputs_embeds=inputs_embeds,
        decoder_token_type_ids=token_type_ids,
        prompt_attention_mask=attention_mask,
        prompt_position_ids=position_ids,
        special_token_ids=[tokenizer.sep_token_id],
        max_length=256,
        num_beams=4,
        return_dict_in_generate=True,
    )['sequences']

    # Findings and impression section:
    findings, impression = model.split_and_decode_sections(output_ids, [tokenizer.sep_token_id, tokenizer.eos_token_id], tokenizer)

    generated.append({'study_id': example['study_id'][0], 'findings': findings[0], 'impression': impression[0]})

pd.DataFrame(generated).to_csv('mimic_cxr_test_set_generated_reports.csv', index=False)

100%|██████████| 3269/3269 [21:32<00:00,  2.53it/s]
